In this notebook, we try and compare 8 different randomization methods.

In each trial:
- The random vector is sampled from either a Uniform[-1, 1] distribution or a standard Normal distribution.
- The random vector is either normalized or not normalized
- the gradient, g, is either normalized or not normalized

Each method is run for 100 times, and we take the average number of iterations.

The hyperparameter c = 1 (descent direction $d = u + \frac{1}{c} g = u + g$)

The best performing combination is:

u ~ uniform [-1, 1] and Normalized, g normalized

2952.75 iteration on average

All results:

- u ~ uniform [-1, 1] and NOT Normalized, g normalized
4063.55

- u ~ uniform [-1, 1] and NOT Normalized, g NOT normalized
3100.03

- u ~ uniform [-1, 1] and Normalized, g normalized
2952.75

- u ~ uniform [-1, 1] and Normalized, g NOT normalized
3145.34


- u ~ N(0, 1) and NOT Normalized, g normalized
3370.39

- u ~ N(0, 1) and NOT Normalized, g NOT normalized
3024.18

- u ~ N(0, 1) and Normalized, g normalized
2961.24

- u ~ N(0, 1) and Normalized, g NOT normalized
2985.61

# Rosenbrock = $10000(y - x^2)^2 + (1 - x)^2$

In [ ]:
# Rosenbrock Function: f_rosenbrock(x) = 10000(y - x^2)^2 + (1 - x)^2
def rosenbrock_function(x):
    return 10000 * (x[1] - x[0]**2)**2 + (1 - x[0])**2

In [ ]:
import numpy as np
from autograd import grad
from autograd import hessian
import autograd.numpy as anp
import matplotlib.pyplot as plt
import time

In [ ]:
from ast import Raise
# @title funcs

# Gradient descent function with backtracking line search and history tracking
def gradient_descent(f, grad_f, x0, alpha=0.3, beta=0.8, precision=1e-6, max_iterations=100000):
    x = x0
    x_history = [x.copy()]  # Initialize history with the starting point
    f_history = [f(x)]
    start_time = time.time()

    for iteration in range(max_iterations):
        gradient = grad_f(x)
        t = 1.0  # Initial step size
        while f(x - t * gradient) > f(x) - alpha * t * np.dot(gradient, gradient):
            t *= beta
        x = x - t * gradient

        x_history.append(x.copy())  # Record the new x value
        f_history.append(f(x))
        if np.linalg.norm(gradient) < precision:
            print("Converged!!")
            break

    if (np.linalg.norm(gradient) > precision):
        print("NOT Converged!!")
    print("grad = ", gradient)
    print("norm = ", np.linalg.norm(gradient))
    time_elapsed = time.time() - start_time
    return x, f_history, np.array(x_history), iteration + 1, time_elapsed

def generate_random_descent_vector(g, c, randtype = "uniform", randnormed = True, gradnormed = True):
    dim = len(g)  # Dimension of the gradient vector


    # if we want to normalize the gradient
    gtemp = g
    if gradnormed == True:
        gtemp = g / np.linalg.norm(g)

    counter = 0
    while True:
        counter += 1
        # Generate a random vector 'u'
        if randtype == "uniform":
            u = np.random.uniform(-1, 1, dim)
        elif randtype == "normal":
            u = np.random.normal(size = dim)
        else:
            raise Exception("randtype = \'", randtype, "\' is not supported.")

        # normalize the random vector
        if randnormed == True:
            u /= np.linalg.norm(u)  # Normalize to make 'u' a unit vector

        # Calculate d = g ⋅ (u + 1/c * g)
        d = u + 1/c * gtemp
        check = np.dot(gtemp, d)

        # Check if the dot product is positive
        if check > 0:
            return d, counter

# Gradient descent function with backtracking line search and history tracking
def random_descent(f, grad_f, x0, rand_scale = 1.0, alpha=0.3, beta=0.8, precision=1e-6, max_iterations=100000, randtype = "uniform", randnormed = True, gradnormed = True):
    x = x0
    x_history = [x.copy()]  # Initialize history with the starting point
    f_history = [f(x)]
    start_time = time.time()

    for iteration in range(max_iterations):
        gradient = grad_f(x)
        d, _ = generate_random_descent_vector(gradient, c = rand_scale, randtype = randtype, randnormed = randnormed, gradnormed = gradnormed)


        # Backtracking line Search
        t = 1.0  # Initial step size
        while f(x - t * d) > f(x) - alpha * t * np.dot(gradient, d):
            t *= beta
        x = x - t * d

        x_history.append(x.copy())  # Record the new x value
        f_history.append(f(x))
        if np.linalg.norm(gradient) < precision:
            break

    if (np.linalg.norm(gradient) > precision):
        print("grad = ", gradient)
        print("norm(grad) = ", np.linalg.norm(gradient))
        print("NOT Converged!!")
    time_elapsed = time.time() - start_time
    return x, f_history, np.array(x_history), iteration + 1, time_elapsed

In [ ]:
# @title initializations

# Automatic gradient computation using Autograd
grad_rosenbrock_function = grad(rosenbrock_function)

# Initial point for the Rosenbrock function
x0_rosenbrock = np.array([-1.2, 1.0])
x0 = x0_rosenbrock

num_trials = 100

In [ ]:
# @title Rand Descent Exp (u ~ uniform [-1, 1] and NOT Normalized, g normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "uniform", randnormed = False, gradnormed = True)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  4016.41
average time (in  100  trials) =  4.3659494638442995


In [ ]:
# @title Rand Descent Exp (u ~ uniform [-1, 1] and NOT Normalized, g NOT normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "uniform", randnormed = False, gradnormed = False)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  2997.75
average time (in  100  trials) =  2.868366494178772


In [ ]:
# @title Rand Descent Exp (u ~ uniform [-1, 1] and Normalized, g normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "uniform", randnormed = True, gradnormed = True)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  3167.4
average time (in  100  trials) =  3.4191416358947753


In [ ]:
# @title Rand Descent Exp (u ~ uniform [-1, 1] and Normalized, g NOT normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "uniform", randnormed = True, gradnormed = False)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  3142.88
average time (in  100  trials) =  3.239545464515686


In [ ]:
# @title Rand Descent Exp (u ~ normal and NOT Normalized, g normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "normal", randnormed = False, gradnormed = True)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  3331.58
average time (in  100  trials) =  3.5431828355789183


In [ ]:
# @title Rand Descent Exp (u ~ normal and NOT Normalized, g NOT normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "normal", randnormed = False, gradnormed = False)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  3081.13
average time (in  100  trials) =  2.98052942276001


In [ ]:
# @title Rand Descent Exp (u ~ normal and Normalized, g normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "normal", randnormed = True, gradnormed = True)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  2881.91
average time (in  100  trials) =  3.037803554534912


In [ ]:
# @title Rand Descent Exp (u ~ normal and Normalized, g NOT normalized)

iter_list = []
time_list = []
for i in range(0, num_trials):
    # Run gradient descent
    solution_rosenbrock_rand, f_hist_rosenbrock_rand, x_hist_rosenbrock_rand, iterations_rosenbrock_rand, time_rosenbrock_rand = random_descent(
        rosenbrock_function, grad_rosenbrock_function, x0_rosenbrock, rand_scale= 1, randtype = "normal", randnormed = True, gradnormed = False)

    iter_list.append(iterations_rosenbrock_rand)
    time_list.append(time_rosenbrock_rand)
print("average iteration count (in ", num_trials, " trials) = ", sum(iter_list) / len(iter_list))
print("average time (in ", num_trials, " trials) = ", sum(time_list) / len(time_list))

average iteration count (in  100  trials) =  3061.53
average time (in  100  trials) =  3.0878482460975647


In [ ]:
# @title Condition number = 250008
#

# Compute the Hessian matrix of the Rosenbrock function
hessian_rosenbrock = hessian(rosenbrock_function)

# Choose a point at which to evaluate the Hessian
x_point = np.array([1.0, 1.0])  # Example point (global minimum)

# Compute the Hessian matrix at the chosen point
H = hessian_rosenbrock(x_point)

# Compute the condition number of the Hessian matrix
condition_number = np.linalg.cond(H)

print("Hessian matrix at point", x_point, "is:")
print(H)
print("Condition number of the Hessian matrix is:", condition_number)


Hessian matrix at point [1. 1.] is:
[[ 80002. -40000.]
 [-40000.  20000.]]
Condition number of the Hessian matrix is: 250008.00009643106
